# Scrape a Listing Page and Its Detail Pages

Collect links to item pages from a listing page, then visit each item page and extract its title and availability into a pandas table. The demo uses the public practice site books.toscrape.com and scrapes politely (robots.txt, User-Agent, delays).

## How to run
Run all cells in order. The result is saved as `items.csv`.

_Rewritten from my original Colab notebook: `download video.ipynb`._

In [ ]:
!pip install -q requests beautifulsoup4 pandas tqdm

In [ ]:
BASE_URL = "https://books.toscrape.com/"
LISTING_URL = BASE_URL + "catalogue/page-1.html"

## A polite fetch helper
Checks `robots.txt`, sends a clear User-Agent and waits between requests.

In [ ]:
import time
import urllib.robotparser
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "learning-scraper/1.0 (educational project)"}
DELAY_SECONDS = 1.5

robots = urllib.robotparser.RobotFileParser()
robots.set_url(urljoin(BASE_URL, "/robots.txt"))
robots.read()

def get_soup(url):
    """Download a page politely and return it parsed with BeautifulSoup."""
    if not robots.can_fetch(HEADERS["User-Agent"], url):
        print("robots.txt does not allow:", url)
        return None
    time.sleep(DELAY_SECONDS)  # be gentle with the server
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

## Step 1: collect item links from the listing page

In [ ]:
soup = get_soup(LISTING_URL)
item_pages = []
for a in soup.select("article.product_pod h3 a"):
    full_url = urljoin(LISTING_URL, a["href"])
    if full_url not in item_pages:
        item_pages.append(full_url)

print(f"Found {len(item_pages)} item pages")
for link in item_pages[:5]:
    print(link)

## Step 2: visit each item page

In [ ]:
from tqdm import tqdm
import pandas as pd

def process_item_page(url):
    try:
        page = get_soup(url)
        title = page.find("h1").get_text(strip=True)
        stock = page.select_one("p.availability").get_text(strip=True)
        return {"url": url, "title": title, "availability": stock}
    except Exception as e:
        return {"url": url, "error": str(e)}

rows = [process_item_page(u) for u in tqdm(item_pages[:10])]  # first 10 to keep it quick
df = pd.DataFrame(rows)
df.to_csv("items.csv", index=False)
df.head()